# State Transitions Notebook

## Data Generator

In [1]:
import numpy as np, pandas as pd
from pathlib import Path

rng = np.random.default_rng(12)
DATA_DIR = Path("../data"); DATA_DIR.mkdir(exist_ok=True)

N_MONTHS = 30                                    # May 2024 to Oct 2026
MONTHS = pd.date_range("2024-05-01", periods=N_MONTHS, freq="MS")
STATES = np.array(["trial", "active", "at_risk", "grace", "lapsed", "won_back"])
TRIAL, ACTIVE, AT_RISK, GRACE, LAPSED, WON_BACK = range(6)
ENG = np.array(["engaged", "casual", "dormant"])
PLATFORMS = np.array(["ios", "android", "web"])
PRICE = np.array([12.99, 69.99])                 # index = plan (0 monthly, 1 annual)
SEASON = {1: 1.35, 2: 1.10, 3: 1.00, 4: 0.95, 5: 0.90, 6: 0.85,
          7: 0.85, 8: 0.90, 9: 1.00, 10: 1.10, 11: 1.15, 12: 1.05}

# users: ~3,000 trial starts a month, growing, seasonal
starts = [rng.poisson(3000 * (1 + 0.012 * t) * SEASON[m.month]) for t, m in enumerate(MONTHS)]
signup_idx = np.repeat(np.arange(N_MONTHS), starts)
N = len(signup_idx)
user_id = np.arange(1, N + 1)
d0 = rng.integers(1, 29, N)                      # trial start day
plat = rng.choice(3, N, p=[.55, .30, .15])
eng = rng.choice(3, N, p=[.35, .45, .20])        # engagement tier
resolve_idx = signup_idx + (d0 > 21)             # 7-day trial can straddle month end

# per-user state, updated month by month
state = np.full(N, TRIAL); state_since = signup_idx.copy()
plan = np.full(N, -1); anchor = np.zeros(N, int)  # anchor = billing day of month
paid_start = np.full(N, -1); ever_paid = np.zeros(N, bool)

ev, bk, eng_rows = [], [], []

def when(t, day):
    return MONTHS[t] + pd.to_timedelta(np.asarray(day) - 1, unit="D")

def emit(idx, t, day, etype, frm, to):
    if len(idx) == 0: return
    ev.append(pd.DataFrame({"user_id": user_id[idx], "event_date": when(t, np.broadcast_to(day, idx.shape)),
                            "event_type": etype, "from_state": frm, "to_state": to, "plan": plan[idx]}))

def book(idx, t, day, btype):
    if len(idx) == 0: return
    bk.append(pd.DataFrame({"user_id": user_id[idx], "booking_date": when(t, day), "plan": plan[idx],
                            "amount": PRICE[plan[idx]], "booking_type": btype}))

def odds(p, mult):
    o = p / (1 - p) * mult
    return o / (1 + o)

P_PAY  = np.array([[.84, .14, .02], [.10, .74, .16], [.03, .20, .77]])   # engagement moves, paying states
P_RISK = np.array([[.55, .35, .10], [.05, .60, .35], [.01, .09, .90]])   # at_risk drifts down faster
P_LAP  = np.array([[.45, .35, .20], [.04, .46, .50], [.01, .04, .95]])   # lapsed mostly go dormant

def step_eng(idx, P):
    cum = P[eng[idx]].cumsum(1)
    eng[idx] = np.minimum((rng.random(len(idx))[:, None] > cum).sum(1), 2)

In [2]:
for t in range(N_MONTHS):
    month = MONTHS[t].month
    new = np.where(signup_idx == t)[0]
    emit(new, t, d0[new], "trial_start", "none", "trial")
    s0 = state.copy()                            # state at start of month; all transitions use this
    fresh = np.zeros(N, bool)
    ten_all = t - paid_start
    bill_all = (t > paid_start) & ((plan == 0) | (ten_all % 12 == 0))

    # 1) trial resolves
    tr = np.where((s0 == TRIAL) & (resolve_idx == t))[0]
    if len(tr):
        mult = np.array([2.6, 1.0, .25])[eng[tr]] * np.array([1.0, .9, 1.1])[plat[tr]] * (1.15 if month == 1 else 1.0)
        conv = rng.random(len(tr)) < odds(.36, mult)
        day = np.where(d0[tr] + 7 > 28, d0[tr] + 7 - 28, d0[tr] + 7)
        c, n = tr[conv], tr[~conv]
        plan[c] = (rng.random(len(c)) < np.array([.30, .22, .35])[plat[c]]).astype(int)
        anchor[c] = day[conv]; paid_start[c] = t; ever_paid[c] = True
        state[c] = ACTIVE; state_since[c] = t
        emit(c, t, day[conv], "trial_convert", "trial", "active"); book(c, t, day[conv], "first_payment")
        state[n] = LAPSED; state_since[n] = t
        emit(n, t, day[~conv], "trial_expire", "trial", "lapsed")

    # 2) paying (active, won_back)
    pay = np.where((s0 == ACTIVE) | (s0 == WON_BACK))[0]
    if len(pay):
        e, pl, ten, bill = eng[pay], plan[pay], ten_all[pay], bill_all[pay]
        wb = s0[pay] == WON_BACK
        off_m = np.array([.010, .040, .105])[e] * np.where(ten <= 2, 1.6, 1.0)
        off_a = np.array([.005, .012, .030])[e] * np.where(np.isin(ten % 12, [10, 11]), 6.0, 1.0)
        p_off = np.minimum(np.where(pl == 0, off_m, off_a) * np.where(wb, 1.4, 1.0), .6)
        p_fail = np.array([.030, .045, .065])[plat[pay]] * np.array([.8, 1.0, 1.3])[e] * bill
        u = rng.random(len(pay)); x = rng.integers(1, 29, len(pay))
        fail = u < p_fail
        off = (~fail) & (u < p_fail + p_off)
        same = off & bill & (x < anchor[pay])    # turned off before billing day: lapses this month
        paid = bill & ~fail & ~same
        book(pay[paid], t, anchor[pay[paid]], "renewal")
        i = pay[fail]; state[i] = GRACE; state_since[i] = t
        emit(i, t, anchor[i], "payment_failed", STATES[s0[i]], "grace")
        i = pay[off]; state[i] = AT_RISK; state_since[i] = t
        emit(i, t, x[off], "auto_renew_off", STATES[s0[i]], "at_risk")
        j = pay[same]; state[j] = LAPSED; state_since[j] = t
        emit(j, t, anchor[j], "lapse", "at_risk", "lapsed")
        g = pay[wb & ~fail & ~off & (ten >= 3)]  # won_back graduates to active after 3 months
        state[g] = ACTIVE; state_since[g] = t
        emit(g, t, rng.integers(1, 29, len(g)), "tenure_graduation", "won_back", "active")

    # 3) at_risk (auto-renew off)
    ar = np.where(s0 == AT_RISK)[0]
    if len(ar):
        e, pl, bill = eng[ar], plan[ar], bill_all[ar]
        p_on = np.where(bill, np.array([.14, .07, .02])[e] * np.where(pl == 1, .6, 1.0), .012)
        on = rng.random(len(ar)) < p_on
        i = ar[on]; state[i] = ACTIVE; state_since[i] = t
        emit(i, t, rng.integers(1, anchor[i] + 1), "auto_renew_on", "at_risk", "active")
        k = ar[on & bill]; book(k, t, anchor[k], "renewal")
        j = ar[bill & ~on]; state[j] = LAPSED; state_since[j] = t
        emit(j, t, anchor[j], "lapse", "at_risk", "lapsed")

    # 4) grace (payment failed)
    gr = np.where(s0 == GRACE)[0]
    if len(gr):
        base = np.array([.60, .58, .48])[plat[gr]]
        if t >= 9: base = np.where(plat[gr] == 0, .74, base)     # iOS billing-retry fix, Feb 2025
        p_rec = np.minimum(base * np.array([1.1, 1.0, .8])[eng[gr]], .92)
        rec = rng.random(len(gr)) < p_rec
        lap = ~rec & (((t - state_since[gr]) >= 2) | (rng.random(len(gr)) < .72))
        i = gr[rec]; d = rng.integers(1, 29, len(i))
        state[i] = ACTIVE; state_since[i] = t; paid_start[i] = t; anchor[i] = d
        emit(i, t, d, "payment_recovered", "grace", "active"); book(i, t, d, "recovered_payment")
        j = gr[lap]; state[j] = LAPSED; state_since[j] = t
        emit(j, t, rng.integers(1, 29, len(j)), "grace_expired", "grace", "lapsed")

    # 5) lapsed: winback hazard decays with months since lapse
    lp = np.where(s0 == LAPSED)[0]
    if len(lp):
        d = t - state_since[lp]
        h = (.055 * np.exp(-.18 * d) + .012) * np.array([2.0, 1.0, .35])[eng[lp]] * np.where(ever_paid[lp], 1.0, .5)
        h = h * (1.5 if month == 1 else 1.0) * (1.3 if t >= 22 else 1.0)   # winback program from Mar 2026
        win = rng.random(len(lp)) < np.minimum(h, .5)
        i = lp[win]; x = rng.integers(1, 29, len(i))
        plan[i] = (rng.random(len(i)) < .20).astype(int)
        anchor[i] = x; paid_start[i] = t; ever_paid[i] = True
        state[i] = WON_BACK; state_since[i] = t; fresh[i] = True
        eng[i] = rng.choice(3, len(i), p=[.30, .60, .10])
        emit(i, t, x, "winback", "lapsed", "won_back"); book(i, t, x, "winback_payment")

    # engagement drifts at month end, then gets logged
    alive = signup_idx <= t
    for grp, P in [((ACTIVE, WON_BACK, GRACE), P_PAY), ((AT_RISK,), P_RISK), ((LAPSED,), P_LAP)]:
        step_eng(np.where(alive & ~fresh & (s0 != TRIAL) & np.isin(state, grp))[0], P)
    idx = np.where(alive)[0]
    eng_rows.append(pd.DataFrame({"user_id": user_id[idx], "month": MONTHS[t], "engagement": ENG[eng[idx]]}))

In [3]:
PLAN_NAME = {0: "monthly", 1: "annual"}
events = pd.concat(ev, ignore_index=True)
events["plan"] = events["plan"].map(PLAN_NAME)
events = pd.concat([events, events.sample(frac=.0015, random_state=12)])        # app double-fires a few events
events = events.sort_values(["user_id", "event_date"], kind="stable").reset_index(drop=True)
events.insert(0, "event_id", np.arange(1, len(events) + 1))

bookings = pd.concat(bk, ignore_index=True)
bookings["plan"] = bookings["plan"].map(PLAN_NAME)
bookings = bookings.sort_values(["user_id", "booking_date"], kind="stable").reset_index(drop=True)
bookings.insert(0, "booking_id", np.arange(1, len(bookings) + 1))

users = pd.DataFrame({"user_id": user_id, "platform": PLATFORMS[plat],
                      "trial_start_date": MONTHS[signup_idx] + pd.to_timedelta(d0 - 1, unit="D")})
engagement = pd.concat(eng_rows, ignore_index=True)

users.to_csv(DATA_DIR / "users.csv", index=False)
events.to_csv(DATA_DIR / "events.csv.gz", index=False)
bookings.to_csv(DATA_DIR / "bookings.csv.gz", index=False)
engagement.to_csv(DATA_DIR / "engagement_monthly.csv.gz", index=False)
print({k: len(v) for k, v in dict(users=users, events=events, bookings=bookings, engagement=engagement).items()})

{'users': 106282, 'events': 285937, 'bookings': 356951, 'engagement': 1560711}


## Import Data

In [4]:
import pandas as pd
D = "../data/"
users = pd.read_csv(D + "users.csv", parse_dates=["trial_start_date"])
events = pd.read_csv(D + "events.csv.gz", parse_dates=["event_date"])
bookings = pd.read_csv(D + "bookings.csv.gz", parse_dates=["booking_date"])
engagement = pd.read_csv(D + "engagement_monthly.csv.gz", parse_dates=["month"])

for name, df in [("users", users), ("events", events), ("bookings", bookings), ("engagement", engagement)]:
    print(f"\n{name}: {df.shape}")
    print(df.dtypes.to_string())
    print("nulls:", df.isna().sum()[lambda s: s > 0].to_dict())
    display(df.head(3))


users: (106282, 3)
user_id                      int64
platform                       str
trial_start_date    datetime64[us]
nulls: {}


,user_id,platform,trial_start_date
0,1,android,2024-05-26
1,2,web,2024-05-12
2,3,ios,2024-05-04



events: (285937, 7)
event_id               int64
user_id                int64
event_date    datetime64[us]
event_type               str
from_state               str
to_state                 str
plan                     str
nulls: {'plan': 170255}


,event_id,user_id,event_date,event_type,from_state,to_state,plan
0,1,1,2024-05-26,trial_start,none,trial,NaN
1,2,1,2024-06-05,trial_convert,trial,active,monthly
2,3,2,2024-05-12,trial_start,none,trial,NaN



bookings: (356951, 6)
booking_id               int64
user_id                  int64
booking_date    datetime64[us]
plan                       str
amount                 float64
booking_type               str
nulls: {}


,booking_id,user_id,booking_date,plan,amount,booking_type
0,1,1,2024-06-05,monthly,12.99,first_payment
1,2,1,2024-07-05,monthly,12.99,renewal
2,3,1,2024-08-05,monthly,12.99,renewal



engagement: (1560711, 3)
user_id                int64
month         datetime64[us]
engagement               str
nulls: {}


,user_id,month,engagement
0,1,2024-05-01,casual
1,2,2024-05-01,engaged
2,3,2024-05-01,engaged


In [5]:
key = ["user_id", "event_date", "event_type"]
print("unique users:", users.user_id.is_unique)
print("events w/o user:", (~events.user_id.isin(users.user_id)).sum())
print("bookings w/o user:", (~bookings.user_id.isin(users.user_id)).sum())
print("duplicate events:", events.duplicated(key).sum())
print("events:", events.event_date.min().date(), "to", events.event_date.max().date())
print("bookings:", bookings.booking_date.min().date(), "to", bookings.booking_date.max().date())
display(events.event_type.value_counts())
display(users.platform.value_counts(normalize=True).round(3))

unique users: True
events w/o user: 0
bookings w/o user: 0
duplicate events: 428
events: 2024-05-01 to 2026-10-28
bookings: 2024-05-08 to 2026-10-28


event_type
trial_start          106435
trial_expire          63820
trial_convert         41530
auto_renew_off        17584
lapse                 15655
payment_failed        11935
winback               10845
payment_recovered      7536
tenure_graduation      6540
grace_expired          3568
auto_renew_on           489
Name: count, dtype: int64

platform
ios        0.552
android    0.297
web        0.151
Name: proportion, dtype: float64

In [6]:
events = events.drop_duplicates(key).reset_index(drop=True)
prev_to = events.groupby("user_id").to_state.shift()
print("broken chains:", (prev_to.notna() & (prev_to != events.from_state)).sum())   # each from_state should equal the prior to_state
display(pd.crosstab(events.from_state, events.to_state))

monthly = bookings.groupby(bookings.booking_date.dt.to_period("M")).amount.agg(bookings="sum", n="size")
display(monthly.iloc[::6])
display(bookings.groupby(["plan", "booking_type"]).amount.agg(["size", "sum"]))

broken chains: 0


to_state,active,at_risk,grace,lapsed,trial,won_back
from_state,,,,,,
active,0,15867,11181,0,0,0
at_risk,489,0,0,15633,0,0
grace,7525,0,0,3560,0,0
lapsed,0,0,0,0,0,10824
none,0,0,0,0,106282,0
trial,41463,0,0,63730,0,0
won_back,6532,1692,731,0,0,0


,bookings,n
booking_date,,
2024-05,22076.22,778
2024-11,98242.02,5698
2025-05,174272.52,10748
2025-11,235353.24,14476
2026-05,302246.94,18906


size         sum
plan    booking_type                         
annual  first_payment       11832   828121.68
        recovered_payment     178    12458.22
        renewal              6257   437927.43
        winback_payment      2155   150828.45
monthly first_payment       29631   384906.69
        recovered_payment    7347    95437.53
        renewal            290882  3778557.18
        winback_payment      8669   112610.31

## Data Alignment